In [ ]:
import mdtraj as md
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
import concurrent.futures
import scipy.spatial.distance


gleap_target = md.load("../20250801_FUS-RRM/2_md-preparation-2/gleap.target.gro")
atom_indices_protein = gleap_target.topology.select("resid 23 to 126")
atom_indices_rna     = gleap_target.topology.select("resid  0 to  22")
atom_indices_rna_docking_surface = gleap_target.topology.select("resid 7 to 16")
atom_index_protein_min = min(atom_indices_protein)

pickle_dir = "./pickle-cache/interactions_vdw"


def calculate_interactions(xtc):
    _interactions = np.zeros((len(atom_indices_protein), len(atom_indices_rna), xtc.n_frames))
    for timestep in range(xtc.n_frames):
        distance_matrix = scipy.spatial.distance.cdist(xtc.xyz[timestep, atom_indices_protein], xtc.xyz[timestep, atom_indices_rna])
        _interactions[((0 < distance_matrix) & (distance_matrix < 0.3)), timestep] += 1
    return _interactions
    

if False:
    def worker(trial):
        print(trial, "start")
        if trial == "cmd":
            interactions = np.zeros((5, len(atom_indices_protein), len(atom_indices_rna), 1800))
            for trial_cmd in range(5):
                xtc = md.load(f"../20250801_FUS-RRM/6_cmd-{trial_cmd+1}/cmd.skip10.whole.target.pbc.xtc", top=gleap_target)[2001::10]
                interactions[trial_cmd] = calculate_interactions(xtc)

            nonzero_indices = interactions.nonzero()
            nonzero_values  = interactions[nonzero_indices]
            pickle.dump([nonzero_indices, nonzero_values], open(f"{pickle_dir}/cmd_interactions.pickle", "wb"))

        else:
            interactions = np.zeros((len(atom_indices_protein), len(atom_indices_rna), 120))
            xtc = md.load(f"../20250801_FUS-RRM/7_pacs-1/trial{trial+1:03}/prd.target.trjcat-all.pbc.skip10.xtc", top=gleap_target)[::10]
            coms_protein             = md.compute_center_of_mass(xtc.atom_slice(atom_indices_protein))
            coms_rna_docking_surface = md.compute_center_of_mass(xtc.atom_slice(atom_indices_rna_docking_surface))
            com_distances = np.linalg.norm(coms_rna_docking_surface-coms_protein, axis=1)
            com_distances_binned = (com_distances*10).astype(int)

            _interactions = calculate_interactions(xtc)
            for timestep, com_distance_binned in enumerate(com_distances_binned):
                interactions[:, :, com_distance_binned] += _interactions[:, :, timestep]

            nonzero_indices = interactions.nonzero()
            nonzero_values  = interactions[nonzero_indices]
            pickle.dump([nonzero_indices, nonzero_values], open(f"{pickle_dir}/pacs_interactions_{trial}.pickle", "wb"))
            pickle.dump(np.bincount(com_distances_binned, minlength=120), open(f"{pickle_dir}/pacs_com_distances_bincount_{trial}.pickle", "wb"))
        print(trial, "completed")

    if not os.path.exists(pickle_dir):
        os.mkdir(pickle_dir)
    trials = ["cmd"]
    trials.extend(range(50))
    for _ in concurrent.futures.ProcessPoolExecutor(max_workers=12).map(worker, trials):
        pass


cmd_interactions = np.zeros((5, len(atom_indices_protein), len(atom_indices_rna), 1800))
nonzero_indices, nonzero_values = pickle.load(open(f"{pickle_dir}/cmd_interactions.pickle", "rb"))
cmd_interactions[nonzero_indices] = nonzero_values

pacs_interactions = np.zeros((50, len(atom_indices_protein), len(atom_indices_rna), 120))
pacs_com_distances_bincount = np.zeros((50, 120))
for trial in range(len(pacs_interactions)):
    nonzero_indices, nonzero_values = pickle.load(open(f"{pickle_dir}/pacs_interactions_{trial}.pickle", "rb"))
    pacs_interactions[trial][nonzero_indices] = nonzero_values
    pacs_com_distances_bincount[trial] = pickle.load(open(f"{pickle_dir}/pacs_com_distances_bincount_{trial}.pickle", "rb"))

pacs_interactions /= pacs_com_distances_bincount[:, np.newaxis, np.newaxis, :]
del pacs_com_distances_bincount

In [ ]:
YMAX = 100

fig, ax = plt.subplots(1,23)
fig.set_size_inches(92, 3)

for residue_index_rna in range(0, 23):
    atom_indices_rna_residue = gleap_target.topology.select(f"resid {residue_index_rna}")
    atom_index_rna_residue_min = min(atom_indices_rna_residue)
    atom_index_rna_residue_max = max(atom_indices_rna_residue)

    for trial in range(len(cmd_interactions)):
        ax[residue_index_rna].plot(cmd_interactions[trial, :, atom_index_rna_residue_min:atom_index_rna_residue_max+1, :].sum(axis=(0, 1)).reshape(-1, 10).mean(axis=1), linewidth=2)

    ax[residue_index_rna].set_title(gleap_target.topology.residue(residue_index_rna))
    ax[residue_index_rna].set_ylim(0, YMAX)

In [ ]:
def make_label_protein(atom):
    return f"{atom.residue.name}{atom.residue.index}-{atom.name}"
    return f"{atom.residue.name}{atom.residue.index+252}-{atom.name}"


atom_indices_protein_rgion = {
    "rrm": list(gleap_target.topology.select("resid  24 to 116")),
    "rgg": list(gleap_target.topology.select("resid 117 to 125")),
}

colors = [
    {
        "rrm": "tab:blue",
        "rgg": "lightskyblue",
    },
    {
        "rrm": "tab:orange",
        "rgg": "#FFBE67",
    },
]

FRACTION_THRESHOLD = 0.1

fig, ax = plt.subplots(1, 23)
fig.set_size_inches(92, 3)

# cmd_interactions_transpose_concat = np.concatenate(cmd_interactions.transpose(0, 3, 1, 2))
cmd_interactions_transpose_concat = np.concatenate(cmd_interactions[:, :, :, ::5].transpose(0, 3, 1, 2))

metastabled_trajs_concat = pickle.load(open(f"./pickle-cache/pca_tetranucl/metastabled_trajs_concat_2.pickle", "rb"))[::50]
for residue_index_rna in range(0, 23):
    atom_idices_rna_residue = gleap_target.topology.select(f"resid {residue_index_rna}")
    atom_index_rna_residue_min = min(atom_idices_rna_residue)
    atom_index_rna_residue_max = max(atom_idices_rna_residue)

    i_bar = 0
    xticks = []
    for atom_index_rna_residue in range(atom_index_rna_residue_min, atom_index_rna_residue_max+1):
        for atom_index_protein in atom_indices_protein:
            fraction_0 = cmd_interactions_transpose_concat[metastabled_trajs_concat==0, atom_index_protein-atom_index_protein_min, atom_index_rna_residue].mean()
            fraction_1 = cmd_interactions_transpose_concat[metastabled_trajs_concat==1, atom_index_protein-atom_index_protein_min, atom_index_rna_residue].mean()

            region = "rrm"
            if atom_index_protein in atom_indices_protein_rgion["rgg"]:
                region = "rgg"
            if fraction_0 > FRACTION_THRESHOLD or fraction_1 > FRACTION_THRESHOLD:
                ax[residue_index_rna].bar(i_bar-0.2, fraction_0, width=0.4, color=colors[0][region])
                ax[residue_index_rna].bar(i_bar+0.2, fraction_1, width=0.4, color=colors[1][region])
                i_bar += 1
                xticks.append(f"{gleap_target.topology.atom(atom_index_rna_residue)}-{make_label_protein(gleap_target.topology.atom(atom_index_protein))}")
                print(f"{gleap_target.topology.atom(atom_index_rna_residue)}-{make_label_protein(gleap_target.topology.atom(atom_index_protein))}", fraction_0, fraction_1)

    ax[residue_index_rna].set_title(gleap_target.topology.residue(residue_index_rna))
    ax[residue_index_rna].set_ylim(0, 1.05)
    ax[residue_index_rna].set_xticks(range(len(xticks)), xticks, rotation=-45)

del cmd_interactions_transpose_concat

In [ ]:
fig, ax = plt.subplots(1, 23)
fig.set_size_inches(92, 3)

pacs_interactions_trial_mean = pacs_interactions.mean(axis=0)

for residue_index_rna in range(0, 23):
    atom_idices_rna_residue = gleap_target.topology.select(f"resid {residue_index_rna}")

    pacs_interactions_trial_mean_rna_sum = pacs_interactions_trial_mean[:, atom_idices_rna_residue].sum(axis=1)

    ax[residue_index_rna].plot(pacs_interactions_trial_mean_rna_sum[atom_indices_protein_rgion["rrm"]-atom_index_protein_min].sum(axis=0))
    ax[residue_index_rna].plot(pacs_interactions_trial_mean_rna_sum[atom_indices_protein_rgion["rgg"]-atom_index_protein_min].sum(axis=0))

    ax[residue_index_rna].set_title(gleap_target.topology.residue(residue_index_rna))
    ax[residue_index_rna].set_ylim(0, 30)

In [ ]:
fig, ax = plt.subplots(len(pacs_interactions), 23)
fig.set_size_inches(92, 3*len(pacs_interactions))

for trial in range(len(pacs_interactions)):
    for residue_index_rna in range(0, 23):
        atom_idices_rna_residue = gleap_target.topology.select(f"resid {residue_index_rna}")

        pacs_interactions_rna_sum = pacs_interactions[trial, :, atom_idices_rna_residue].sum(axis=0)

        ax[trial, residue_index_rna].plot(pacs_interactions_rna_sum[atom_indices_protein_rgion["rrm"]-atom_index_protein_min].sum(axis=0))
        ax[trial, residue_index_rna].plot(pacs_interactions_rna_sum[atom_indices_protein_rgion["rgg"]-atom_index_protein_min].sum(axis=0))

        ax[trial, residue_index_rna].set_title(f"{trial}-{gleap_target.topology.residue(residue_index_rna)}")
        ax[trial, residue_index_rna].set_ylim(0, 30)

In [ ]:
trial_groups_initial = [
    [1, 2, 3, 4, 5, 6, 7, 9, 24, 25, 26, 27, 28, 29],
    [],
]
trial_groups_initial[1] = [trial for trial in range(50) if not trial in trial_groups_initial[0]]


fig, ax = plt.subplots(1, 23)
fig.set_size_inches(92, 3)

for i_trial_group, trial_group_trials in enumerate(trial_groups_initial):
    pacs_interactions_trial_sum = np.zeros((len(atom_indices_protein), len(atom_indices_rna), 120))
    for trial in trial_group_trials:
        pacs_interactions_trial_sum += pacs_interactions[trial]
    pacs_interactions_trial_mean = pacs_interactions_trial_sum / len(trial_group_trials)
    
    for residue_index_rna in range(0, 23):
        atom_idices_rna_residue = gleap_target.topology.select(f"resid {residue_index_rna}")
    
        pacs_interactions_trial_mean_rna_sum = pacs_interactions_trial_mean[:, atom_idices_rna_residue].sum(axis=1)
    
        ax[residue_index_rna].plot(pacs_interactions_trial_mean_rna_sum[atom_indices_protein_rgion["rrm"]-atom_index_protein_min].sum(axis=0), c=colors[i_trial_group]["rrm"], linestyle="-", linewidth=2)
        ax[residue_index_rna].plot(pacs_interactions_trial_mean_rna_sum[atom_indices_protein_rgion["rgg"]-atom_index_protein_min].sum(axis=0), c=colors[i_trial_group]["rgg"], linestyle="-", linewidth=2)
    
        ax[residue_index_rna].set_title(gleap_target.topology.residue(residue_index_rna))
        ax[residue_index_rna].set_xlim(15, 60)
        ax[residue_index_rna].set_ylim(0, 30)

In [ ]:
trial_groups_initial = [
    [7, 25, 26, 28, 29],
    [],
]
trial_groups_initial[1] = [trial for trial in range(50) if not trial in trial_groups_initial[0]]


fig, ax = plt.subplots(1, 23)
fig.set_size_inches(92, 3)

for i_trial_group, trial_group_trials in enumerate(trial_groups_initial):
    pacs_interactions_trial_sum = np.zeros((len(atom_indices_protein), len(atom_indices_rna), 120))
    for trial in trial_group_trials:
        pacs_interactions_trial_sum += pacs_interactions[trial]
    pacs_interactions_trial_mean = pacs_interactions_trial_sum / len(trial_group_trials)
    
    for residue_index_rna in range(0, 23):
        atom_idices_rna_residue = gleap_target.topology.select(f"resid {residue_index_rna}")
    
        pacs_interactions_trial_mean_rna_sum = pacs_interactions_trial_mean[:, atom_idices_rna_residue].sum(axis=1)
    
        ax[residue_index_rna].plot(pacs_interactions_trial_mean_rna_sum[atom_indices_protein_rgion["rrm"]-atom_index_protein_min].sum(axis=0), c=colors[i_trial_group]["rrm"], linestyle="-", linewidth=2)
        ax[residue_index_rna].plot(pacs_interactions_trial_mean_rna_sum[atom_indices_protein_rgion["rgg"]-atom_index_protein_min].sum(axis=0), c=colors[i_trial_group]["rgg"], linestyle="-", linewidth=2)
    
        ax[residue_index_rna].set_title(gleap_target.topology.residue(residue_index_rna))
        ax[residue_index_rna].set_xlim(15, 60)
        ax[residue_index_rna].set_ylim(0, 30)